# Week 2 — YOLO11 Object Detection Training

Fine-tune a pretrained YOLO11n detector on the public COCO128 sample dataset using Ultralytics (PyTorch). COCO128 is a small pipeline-learning dataset, not a production benchmark. Run cells from top to bottom.


## 1. Install and import dependencies


In [ ]:
%pip install -q ultralytics opencv-python pandas matplotlib


In [ ]:
import json, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Experiment configuration

The baseline uses 30 epochs, 640-pixel images, and batch size 8. If memory is insufficient, lower BATCH to 4 or 2. For a quick smoke test, set EPOCHS to 2–3; do not report a smoke test as the full experiment.


In [ ]:
MODEL = 'yolo11n.pt'
DATA = 'coco128.yaml'  # Ultralytics downloads/prepares this dataset on first use
EPOCHS = 30
IMAGE_SIZE = 640
BATCH = 8
WORKERS = 2
PATIENCE = 10
RUN_PROJECT = 'runs/week2_notebook'
RUN_NAME = 'yolo11n_coco128'


## 3. Train the detector

The pretrained checkpoint is fine-tuned. Ultralytics handles model-specific resizing, normalization, label transforms, loss computation, and training-time detection augmentation.


In [ ]:
model = YOLO(MODEL)
start_time = time.perf_counter()
train_results = model.train(
    data=DATA, epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
    workers=WORKERS, patience=PATIENCE, seed=SEED,
    project=RUN_PROJECT, name=RUN_NAME, exist_ok=True,
    plots=True, save=True,
)
training_seconds = time.perf_counter() - start_time
run_dir = Path(model.trainer.save_dir)
print('Run directory:', run_dir)
print('Training time (seconds):', round(training_seconds, 2))


## 4. Inspect training history

The CSV is generated by the training run. Column names can vary slightly by Ultralytics version, so inspect them before interpreting the curves.


In [ ]:
history = pd.read_csv(run_dir / 'results.csv')
history.columns = [c.strip() for c in history.columns]
display(history.tail())
print('Recorded epochs:', len(history))
loss_columns = [c for c in history.columns if 'loss' in c.lower()]
if loss_columns:
    history[loss_columns].plot(figsize=(10, 5), title='Training and validation losses')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(True); plt.show()
else:
    print('No loss columns found:', history.columns.tolist())


## 5. Validate the best checkpoint

Precision, recall, and mAP are more suitable than plain classification accuracy for object detection.


In [ ]:
best_weights = run_dir / 'weights' / 'best.pt'
if not best_weights.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_weights}')
best_model = YOLO(str(best_weights))
validation = best_model.val(data=DATA, imgsz=IMAGE_SIZE, batch=BATCH, plots=True)
metrics = {}
for key, value in validation.results_dict.items():
    try:
        metrics[key] = float(value)
    except (TypeError, ValueError):
        metrics[key] = str(value)
print(json.dumps(metrics, indent=2))


## 6. Save a run summary

Keep the actual values produced by this run. Do not replace them with illustrative values. Record the computer/GPU used when writing the final report.


In [ ]:
summary = {
    'model': MODEL, 'dataset': DATA, 'epochs_requested': EPOCHS,
    'epochs_recorded': int(len(history)), 'image_size': IMAGE_SIZE,
    'batch_size': BATCH, 'seed': SEED, 'pytorch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'training_seconds': round(training_seconds, 2),
    'validation_metrics': metrics,
    'warning': 'COCO128 is a small sample dataset, not a production benchmark.'
}
summary_path = run_dir / 'week2_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved:', summary_path)
print(json.dumps(summary, indent=2))


## 7. Analysis checklist

- Did training losses decrease?
- Did validation losses and mAP improve, plateau, or become unstable?
- Are there classes with weaker precision/recall?
- Does the confusion matrix show systematic class confusion?
- Does the model miss small, blurred, occluded, or low-light objects?
- What hardware, image size, and batch size were used?

Do not infer generalization from COCO128 alone. For the next iteration, use a larger dataset and an independent split, ideally grouped by camera or video sequence for road-scene data.
